# Agent 4 — AI Summarization

Reads all KPI validation runs with ai_enrichment_status = PENDING,
assembles an evidence bundle for each run, calls the AI model,
and writes the enrichment back to the results tables.

Failure categories produced:
  CALCULATION_DRIFT    KPI formula producing wrong values
  FRESHNESS_LAG        KPI not updated for expected period
  SEMANTIC_MISMATCH    KPI definition changed without notice
  UNEXPLAINED_ANOMALY  Cannot explain the swing from history

Reads from:
  uc4_kpi_validation_run_b
  uc4_kpi_validation_results_b
  uc4_kpi_baseline_metrics_b
  uc4_kpi_registry_b

Updates:
  uc4_kpi_validation_results_b  (ai fields)
  uc4_kpi_validation_run_b      (ai_enrichment_status)

## Cell 2: Imports and Configuration

In [0]:
import requests
import json
import uuid
from datetime import datetime
from pyspark.sql import functions as F
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# ── Catalog and Schema ─────────────────────────────────────────────────────

dbutils.widgets.text("catalog", "learning_dev", "Target Catalog")
dbutils.widgets.text("schema", "s819534", "Target Schema")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA  = dbutils.widgets.get("schema")

# ── Table References ───────────────────────────────────────────────────────

KPI_VALIDATION_RUN     = f"{CATALOG}.{SCHEMA}.uc4_kpi_validation_run_b"
KPI_VALIDATION_RESULTS = f"{CATALOG}.{SCHEMA}.uc4_kpi_validation_results_b"
KPI_BASELINE_METRICS   = f"{CATALOG}.{SCHEMA}.uc4_kpi_baseline_metrics_b"
KPI_REGISTRY           = f"{CATALOG}.{SCHEMA}.uc4_kpi_registry_b"

# ── Model Configuration ────────────────────────────────────────────────────

MODEL_ENDPOINT = "databricks-gpt-oss-20b"
MODEL_VERSION  = "gpt-oss-20b-080525"
MAX_TOKENS     = 4096
TEMPERATURE    = 0.1

# ── Workspace Connection ───────────────────────────────────────────────────

WORKSPACE_HOST  = spark.conf.get("spark.databricks.workspaceUrl")
WORKSPACE_TOKEN = (
    dbutils.notebook.entry_point
           .getDbutils().notebook().getContext()
           .apiToken().get()
)

HEADERS = {
    "Authorization": f"Bearer {WORKSPACE_TOKEN}",
    "Content-Type" : "application/json"
}

MODEL_URL = (
    f"https://{WORKSPACE_HOST}"
    f"/serving-endpoints/{MODEL_ENDPOINT}/invocations"
)

VALID_FAILURE_CATEGORIES = {
    "CALCULATION_DRIFT",
    "FRESHNESS_LAG",
    "SEMANTIC_MISMATCH",
    "UNEXPLAINED_ANOMALY",
    "COMBINED_ISSUE"
}

print("=" * 60)
print("AGENT 4 — AI SUMMARIZATION")
print("=" * 60)
print(f"Model endpoint : {MODEL_ENDPOINT}")
print(f"Model URL      : {MODEL_URL}")
print(f"Token length   : {len(WORKSPACE_TOKEN)}")
print(f"Started at     : {datetime.now()}")
print("=" * 60)

## Cell 3: Load Pending Runs

In [0]:
def load_pending_runs() -> list:
    """
    Loads all KPI validation runs where AI enrichment is pending.
    """
    df = (
        spark.table(KPI_VALIDATION_RUN)
             .filter(F.col("ai_enrichment_status") == "PENDING")
             .orderBy("business_date", "gold_fully_qualified_name")
    )

    rows = [row.asDict() for row in df.collect()]

    print(f"Pending runs found: {len(rows)}")
    for r in rows:
        print(
            f"  {r['business_date']}  "
            f"{r['gold_fully_qualified_name'].split('.')[-1]:<40}  "
            f"Status: {r['overall_status']}"
        )

    return rows


pending_runs = load_pending_runs()

if not pending_runs:
    print("\nNo pending runs found. Nothing to process.")
    NO_PENDING_RUNS = True
else:
    NO_PENDING_RUNS = False
    print(f"\nProceeding to process {len(pending_runs)} run(s).")

## Cell 4: Evidence Bundle Assembly

In [0]:
def load_failed_kpi_checks(validation_run_id: str) -> list:
    """
    Loads all WARNING and CRITICAL KPI check results for a run.
    """
    df = (
        spark.table(KPI_VALIDATION_RESULTS)
             .filter(
                 (F.col("validation_run_id") == validation_run_id) &
                 (F.col("check_status").isin(["WARNING", "CRITICAL"]))
             )
             .orderBy(
                 F.when(F.col("check_status") == "CRITICAL", 1)
                  .otherwise(2),
                 "kpi_name",
                 "check_type"
             )
    )
    return [row.asDict() for row in df.collect()]


def load_kpi_definitions(gold_fqn: str) -> list:
    """
    Loads KPI registry definitions for context.
    """
    df = (
        spark.table(KPI_REGISTRY)
             .filter(
                 (F.col("gold_fully_qualified_name") == gold_fqn) &
                 (F.col("is_active") == True)
             )
    )
    return [row.asDict() for row in df.collect()]


def load_kpi_baselines(gold_fqn: str, failed_checks: list) -> list:
    """
    Loads baselines for failed KPIs to provide historical context.
    """
    baselines = []
    seen_kpis = set()

    for check in failed_checks:
        kpi_name   = check["kpi_name"]
        kpi_column = check["kpi_column"]
        key        = f"{kpi_name}_{kpi_column}"

        if key in seen_kpis:
            continue
        seen_kpis.add(key)

        rows = (
            spark.table(KPI_BASELINE_METRICS)
                 .filter(
                     (F.col("gold_fully_qualified_name") == gold_fqn) &
                     (F.col("kpi_name") == kpi_name) &
                     (F.col("kpi_column") == kpi_column)
                 )
                 .orderBy(F.col("baseline_date").desc())
                 .limit(1)
                 .collect()
        )

        if rows:
            b = rows[0].asDict()
            baselines.append({
                "kpi_name"      : kpi_name,
                "kpi_column"    : kpi_column,
                "baseline_mean" : b.get("mean_value"),
                "baseline_stddev": b.get("stddev_value"),
                "baseline_min"  : b.get("min_value"),
                "baseline_max"  : b.get("max_value"),
                "sample_count"  : b.get("sample_count"),
                "baseline_status": b.get("baseline_status")
            })

    return baselines


def assemble_evidence_bundle(
    run_record    : dict,
    failed_checks : list,
    baselines     : list,
    kpi_defs      : list
) -> dict:
    """
    Assembles the complete evidence bundle for AI summarization.
    """
    gold_fqn   = run_record["gold_fully_qualified_name"]
    table_name = gold_fqn.split(".")[-1]

    cleaned_checks = []
    for check in failed_checks:
        detail = {}
        if check.get("check_detail"):
            try:
                detail = json.loads(check["check_detail"])
            except (json.JSONDecodeError, TypeError):
                detail = {}

        cleaned_checks.append({
            "kpi_name"      : check["kpi_name"],
            "kpi_column"    : check["kpi_column"],
            "check_type"    : check["check_type"],
            "check_status"  : check["check_status"],
            "observed_value": check["observed_value"],
            "expected_value": check["expected_value"],
            "deviation_pct" : check["deviation_percentage"],
            "warn_threshold": check["threshold_warning"],
            "crit_threshold": check["threshold_critical"],
            "check_detail"  : detail
        })

    # Build KPI definition map for context
    kpi_def_map = {}
    for kpi in kpi_defs:
        kpi_def_map[kpi["kpi_name"]] = {
            "description"      : kpi.get("kpi_description"),
            "formula"          : kpi.get("formula_expression"),
            "min_value"        : kpi.get("min_value"),
            "max_value"        : kpi.get("max_value")
        }

    return {
        "context": {
            "validation_run_id"      : run_record["validation_run_id"],
            "gold_fully_qualified_name": gold_fqn,
            "table_name"             : table_name,
            "business_date"          : str(run_record["business_date"]),
            "overall_status"         : run_record["overall_status"],
            "total_kpis_checked"     : run_record["total_kpis_checked"],
            "total_kpis_passed"      : run_record["total_kpis_passed"],
            "total_kpis_warned"      : run_record["total_kpis_warned"],
            "total_kpis_critical"    : run_record["total_kpis_critical"]
        },
        "failed_kpi_checks"   : cleaned_checks,
        "kpi_definitions"     : kpi_def_map,
        "baseline_context"    : baselines
    }


print("Evidence bundle functions loaded.")

## Cell 5: Prompt Builder

In [0]:
def build_prompt(evidence_bundle: dict) -> str:
    """
    Builds the AI prompt for Agent 4 KPI validation summarization.

    The prompt focuses on explaining KPI failures in business terms
    and classifying them into one of four categories specific to
    Gold layer issues.
    """
    ctx          = evidence_bundle["context"]
    table_name   = ctx["table_name"]
    business_date= ctx["business_date"]
    overall      = ctx["overall_status"]
    failed_checks= evidence_bundle["failed_kpi_checks"]
    kpi_defs     = evidence_bundle["kpi_definitions"]
    baselines    = evidence_bundle["baseline_context"]

    # Format failed KPI checks
    checks_text = ""
    for i, check in enumerate(failed_checks, 1):
        kpi_def = kpi_defs.get(check["kpi_name"], {})
        checks_text += f"""
  KPI Check {i}:
    KPI Name       : {check['kpi_name']}
    KPI Column     : {check['kpi_column']}
    Check Type     : {check['check_type']}
    Status         : {check['check_status']}
    Observed Value : {check['observed_value']}
    Expected Value : {check['expected_value']}
    Deviation      : {check['deviation_pct']}%
    Warning at     : {check['warn_threshold']}
    Critical at    : {check['crit_threshold']}
    KPI Formula    : {kpi_def.get('formula', 'N/A')}
    KPI Description: {kpi_def.get('description', 'N/A')}
    Detail         : {json.dumps(check['check_detail'], indent=6)}
"""

    # Format baseline context
    baseline_text = ""
    for b in baselines:
        baseline_text += f"""
  {b['kpi_name']} ({b['kpi_column']}):
    Baseline mean   : {b['baseline_mean']}
    Baseline stddev : {b['baseline_stddev']}
    Baseline min    : {b['baseline_min']}
    Baseline max    : {b['baseline_max']}
    Sample count    : {b['sample_count']}
    Baseline status : {b['baseline_status']}
"""

    prompt = f"""You are a senior data analyst reviewing Gold layer KPI
validation results for a retail data platform.

CRITICAL SCOPE CONSTRAINT:
You are analyzing ONE Gold table only: {table_name}
Business date: {business_date}
Do not reference any other table in your response.

GOLD TABLE INFORMATION:
  Table name    : {table_name}
  Business date : {business_date}
  Overall status: {overall}
  KPIs checked  : {ctx['total_kpis_checked']}
  KPIs failed   : {len(failed_checks)}

FAILED KPI CHECKS FOR {table_name} ON {business_date}:
{checks_text}

HISTORICAL BASELINE CONTEXT FOR {table_name}:
{baseline_text}

FAILURE CATEGORY DEFINITIONS:
  CALCULATION_DRIFT   : The KPI formula is producing mathematically
                        wrong values. The calculation logic itself has
                        an error or the underlying data columns have
                        changed in a way that breaks the formula.

  FRESHNESS_LAG       : The KPI values are correct but stale.
                        The Gold table has not been refreshed for
                        the expected business period.

  SEMANTIC_MISMATCH   : The KPI definition or business rule has
                        changed without a corresponding update to
                        the validation thresholds or formula.
                        The KPI is being measured differently than
                        what the validation expects.

  UNEXPLAINED_ANOMALY : The KPI value is statistically anomalous
                        but no clear calculation error, freshness
                        issue, or semantic change can be identified
                        from the available evidence.

  COMBINED_ISSUE      : Multiple distinct failure types affecting
                        the same KPI or multiple KPIs simultaneously.

INSTRUCTIONS:
Analyze the failed KPI checks and produce a JSON response.
Every sentence must refer to {table_name} and {business_date} only.

{{
  "ai_summary": "2-3 sentences explaining what KPI issues were
                  detected in {table_name} on {business_date}.
                  Name the specific KPIs affected, the observed
                  values versus expected values, and the business
                  impact. Be precise about the numbers.",

  "ai_recommended_action": "1-2 specific steps the data engineering
                             or finance team should take immediately.
                             Reference {table_name}, the specific KPI
                             name, and {business_date} explicitly.",

  "ai_failure_category": "Exactly one of: CALCULATION_DRIFT,
                           FRESHNESS_LAG, SEMANTIC_MISMATCH,
                           UNEXPLAINED_ANOMALY, COMBINED_ISSUE",

  "ai_confidence_score": 0.0 to 1.0
}}

Respond with valid JSON only. No markdown. No text outside the JSON."""

    return prompt


print("Prompt builder loaded.")

## Cell 6: AI Model Caller and Response Parser

In [0]:
def extract_content_from_response(response_data: dict) -> str:
    """
    Extracts text content from model response.
    Handles the gpt-oss-20b reasoning block format.
    """
    choices = response_data.get("choices", [])
    if not choices:
        return ""

    first_choice = choices[0]
    message      = first_choice.get("message", {})
    content      = message.get("content", "")

    if isinstance(content, str):
        return content

    if isinstance(content, list):
        text_parts     = []
        reasoning_text = []

        for block in content:
            if not isinstance(block, dict):
                if isinstance(block, str):
                    text_parts.append(block)
                continue

            block_type = block.get("type", "")

            if block_type == "text":
                text = block.get("text", "")
                if text:
                    text_parts.append(text)
            elif block_type == "output_text":
                text = block.get("text", "")
                if text:
                    text_parts.append(text)
            elif block_type == "reasoning":
                for s in block.get("summary", []):
                    if isinstance(s, dict):
                        s_text = s.get("text", "")
                        if s_text:
                            reasoning_text.append(s_text)

        if text_parts:
            return " ".join(text_parts)
        if reasoning_text:
            return " ".join(reasoning_text)

    text = first_choice.get("text", "")
    if text:
        return text

    return ""


def call_ai_model(prompt: str, invocation_id: str) -> dict:
    """
    Calls the AI model and returns parsed response.
    """
    payload = {
        "messages": [
            {
                "role"   : "system",
                "content": (
                    "You are a senior data analyst for a retail platform. "
                    "Always respond with valid JSON only. "
                    "Never include markdown or explanation outside the JSON."
                )
            },
            {
                "role"   : "user",
                "content": prompt
            }
        ],
        "max_tokens" : MAX_TOKENS,
        "temperature": TEMPERATURE
    }

    try:
        response = requests.post(
            MODEL_URL,
            headers=HEADERS,
            json=payload,
            timeout=180
        )

        if response.status_code != 200:
            return {
                "success"     : False,
                "error"       : f"HTTP {response.status_code}: {response.text[:500]}",
                "raw_response": response.text
            }

        response_data = response.json()
        raw_content   = extract_content_from_response(response_data)

        if not isinstance(raw_content, str):
            raw_content = str(raw_content)

        if not raw_content.strip():
            return {
                "success"     : False,
                "error"       : "Empty content from model",
                "raw_response": str(response_data)
            }

        parsed = parse_ai_response(raw_content)

        if parsed["success"]:
            return {
                "success"              : True,
                "ai_summary"           : parsed["ai_summary"],
                "ai_recommended_action": parsed["ai_recommended_action"],
                "ai_failure_category"  : parsed["ai_failure_category"],
                "ai_confidence_score"  : parsed["ai_confidence_score"],
                "raw_response"         : raw_content
            }
        else:
            return {
                "success"     : False,
                "error"       : parsed["error"],
                "raw_response": raw_content
            }

    except requests.exceptions.Timeout:
        return {
            "success"     : False,
            "error"       : "Model call timed out after 180 seconds",
            "raw_response": None
        }
    except Exception as e:
        logger.error(f"Model call exception: {str(e)}")
        return {
            "success"     : False,
            "error"       : str(e),
            "raw_response": None
        }


def parse_ai_response(raw_content: str) -> dict:
    """
    Parses raw model response into structured fields.
    """
    if not raw_content or not raw_content.strip():
        return {"success": False, "error": "Empty response"}

    content = raw_content.strip()

    if "```json" in content:
        content = content.split("```json")[1]
        if "```" in content:
            content = content.split("```")[0]
    elif "```" in content:
        parts = content.split("```")
        if len(parts) >= 2:
            content = parts[1]

    content = content.strip()
    start   = content.find("{")
    end     = content.rfind("}") + 1

    if start == -1 or end == 0:
        return {
            "success": False,
            "error"  : f"No JSON found. Preview: {content[:200]}"
        }

    try:
        parsed = json.loads(content[start:end])
    except json.JSONDecodeError as e:
        return {
            "success": False,
            "error"  : f"JSON parse error: {str(e)}"
        }

    required = [
        "ai_summary",
        "ai_recommended_action",
        "ai_failure_category",
        "ai_confidence_score"
    ]

    missing = [f for f in required if f not in parsed]
    if missing:
        return {
            "success": False,
            "error"  : f"Missing fields: {missing}"
        }

    category = str(parsed.get("ai_failure_category", "")).upper().strip()
    if category not in VALID_FAILURE_CATEGORIES:
        category = "COMBINED_ISSUE"

    try:
        confidence = float(parsed.get("ai_confidence_score", 0.5))
        confidence = max(0.0, min(1.0, confidence))
    except (TypeError, ValueError):
        confidence = 0.5

    return {
        "success"              : True,
        "ai_summary"           : str(parsed.get("ai_summary", "")).strip(),
        "ai_recommended_action": str(parsed.get("ai_recommended_action", "")).strip(),
        "ai_failure_category"  : category,
        "ai_confidence_score"  : confidence
    }


print("AI model caller and parser loaded.")

## Cell 7: Write AI Enrichment

In [0]:
def write_ai_enrichment(
    validation_run_id  : str,
    ai_summary         : str,
    ai_recommended_action: str,
    ai_failure_category: str,
    ai_confidence_score: float,
    ai_invocation_id   : str,
    enriched_timestamp : datetime
) -> bool:
    """
    Updates all failed KPI check rows with AI enrichment fields.
    """
    try:
        safe_summary = ai_summary.replace("'", "\\'")
        safe_action  = ai_recommended_action.replace("'", "\\'")

        spark.sql(f"""
            UPDATE {KPI_VALIDATION_RESULTS}
            SET
                ai_summary              = '{safe_summary}',
                ai_recommended_action   = '{safe_action}',
                ai_failure_category     = '{ai_failure_category}',
                ai_confidence_score     = {ai_confidence_score},
                ai_model_version        = '{MODEL_VERSION}',
                ai_invocation_id        = '{ai_invocation_id}',
                ai_enriched_timestamp   = '{enriched_timestamp}'
            WHERE validation_run_id     = '{validation_run_id}'
              AND check_status IN ('WARNING', 'CRITICAL')
        """)

        logger.info(f"AI enrichment written for run {validation_run_id}")
        return True

    except Exception as e:
        logger.error(f"Failed to write AI enrichment: {str(e)}")
        return False


def update_run_ai_status(
    validation_run_id  : str,
    ai_status          : str,
    enriched_timestamp : datetime
) -> bool:
    """
    Updates the validation run AI enrichment status.
    """
    try:
        spark.sql(f"""
            UPDATE {KPI_VALIDATION_RUN}
            SET
                ai_enrichment_status    = '{ai_status}',
                ai_enrichment_timestamp = '{enriched_timestamp}'
            WHERE validation_run_id     = '{validation_run_id}'
        """)
        return True

    except Exception as e:
        logger.error(f"Failed to update run status: {str(e)}")
        return False


print("Write functions loaded.")

## Cell 8: Main Processing Loop

In [0]:
runs_processed   = 0
runs_succeeded   = 0
runs_failed      = 0
invocation_log   = []

if NO_PENDING_RUNS:
    print("Skipping processing — no pending runs found.")
else:
    print(f"Processing {len(pending_runs)} pending run(s)...\n")

    for run_record in pending_runs:

        run_id     = run_record["validation_run_id"]
        gold_fqn   = run_record["gold_fully_qualified_name"]
        table_name = gold_fqn.split(".")[-1]
        biz_date   = run_record["business_date"]

        print(f"{'='*60}")
        print(f"Run ID     : {run_id}")
        print(f"Table      : {table_name}")
        print(f"Date       : {biz_date}")
        print(f"{'='*60}")

        invocation_id = str(uuid.uuid4())

        try:
            # ── Load failed checks ─────────────────────────────────────────
            failed_checks = load_failed_kpi_checks(run_id)
            print(f"Failed KPI checks : {len(failed_checks)}")

            if not failed_checks:
                print("No failed checks. Skipping AI call.")
                update_run_ai_status(run_id, "NOT_REQUIRED", datetime.now())
                runs_processed += 1
                continue

            # ── Load supporting context ────────────────────────────────────
            kpi_defs  = load_kpi_definitions(gold_fqn)
            baselines = load_kpi_baselines(gold_fqn, failed_checks)

            print(f"KPI definitions   : {len(kpi_defs)}")
            print(f"Baseline records  : {len(baselines)}")

            # ── Assemble evidence bundle ───────────────────────────────────
            evidence_bundle = assemble_evidence_bundle(
                run_record, failed_checks, baselines, kpi_defs
            )
            print(f"Evidence bundle assembled.")

            # ── Build prompt ───────────────────────────────────────────────
            prompt = build_prompt(evidence_bundle)
            print(f"Prompt length     : {len(prompt)} characters")

            # ── Call AI model ──────────────────────────────────────────────
            print(f"Calling model     : {MODEL_ENDPOINT}")
            ai_result = call_ai_model(prompt, invocation_id)

            if ai_result["success"]:
                print(f"Model succeeded.")
                print(f"Category          : {ai_result['ai_failure_category']}")
                print(f"Confidence        : {ai_result['ai_confidence_score']}")
                print(f"Summary preview   : {ai_result['ai_summary'][:100]}...")

                enriched_at   = datetime.now()
                write_success = write_ai_enrichment(
                    validation_run_id      = run_id,
                    ai_summary             = ai_result["ai_summary"],
                    ai_recommended_action  = ai_result["ai_recommended_action"],
                    ai_failure_category    = ai_result["ai_failure_category"],
                    ai_confidence_score    = ai_result["ai_confidence_score"],
                    ai_invocation_id       = invocation_id,
                    enriched_timestamp     = enriched_at
                )

                if write_success:
                    update_run_ai_status(run_id, "COMPLETED", enriched_at)
                    runs_succeeded += 1
                    print(f"Status            : COMPLETED ✅")
                else:
                    update_run_ai_status(run_id, "FAILED", datetime.now())
                    runs_failed += 1
                    print(f"Status            : FAILED (write error) ❌")

            else:
                print(f"Model failed: {ai_result.get('error')}")
                update_run_ai_status(run_id, "FAILED", datetime.now())
                runs_failed += 1

            invocation_log.append({
                "run_id"       : run_id,
                "table_name"   : table_name,
                "business_date": str(biz_date),
                "invocation_id": invocation_id,
                "success"      : ai_result["success"],
                "category"     : ai_result.get("ai_failure_category"),
                "confidence"   : ai_result.get("ai_confidence_score"),
                "error"        : ai_result.get("error")
            })

        except Exception as e:
            print(f"UNEXPECTED ERROR: {str(e)}")
            import traceback
            traceback.print_exc()
            update_run_ai_status(run_id, "FAILED", datetime.now())
            runs_failed += 1

        finally:
            runs_processed += 1
            print()

    print(f"Processing complete.")
    print(f"  Processed : {runs_processed}")
    print(f"  Succeeded : {runs_succeeded}")
    print(f"  Failed    : {runs_failed}")

## Cell 9: Verify Results

In [0]:
print("=" * 60)
print("AI ENRICHMENT VERIFICATION")
print("=" * 60)

# Run status
print("\n1. KPI Validation Run AI Status:\n")
spark.sql(f"""
    SELECT
        business_date,
        SUBSTRING_INDEX(gold_fully_qualified_name, '.', -1) AS gold_table,
        overall_status,
        ai_enrichment_status,
        ai_enrichment_timestamp
    FROM {KPI_VALIDATION_RUN}
    ORDER BY business_date, gold_fully_qualified_name
""").show(20, truncate=False)

# AI enriched results
print("\n2. AI Enriched KPI Check Results:\n")
spark.sql(f"""
    SELECT
        r.business_date,
        SUBSTRING_INDEX(r.gold_fully_qualified_name, '.', -1) AS gold_table,
        r.kpi_name,
        r.check_type,
        r.check_status,
        r.ai_failure_category,
        ROUND(r.ai_confidence_score, 2)                        AS confidence,
        r.ai_summary,
        r.ai_recommended_action
    FROM {KPI_VALIDATION_RESULTS} r
    WHERE r.ai_summary IS NOT NULL
    ORDER BY
        r.business_date,
        r.gold_fully_qualified_name,
        r.kpi_name
""").show(50, truncate=False)

# Invocation log
print("\n3. AI Invocation Log:\n")
for entry in invocation_log:
    status = "✅" if entry["success"] else "❌"
    print(
        f"  {status} {entry['business_date']}  "
        f"{entry['table_name']:<40}  "
        f"Category: {entry['category']}  "
        f"Confidence: {entry['confidence']}"
    )
    if not entry["success"]:
        print(f"     Error: {entry['error']}")

## Cell 10: Final Summary

In [0]:
print("\n" + "=" * 60)
print("AGENT 4 AI SUMMARIZATION COMPLETE")
print("=" * 60)

if NO_PENDING_RUNS:
    print("Runs processed   : 0 (no pending runs found)")
else:
    print(f"Runs processed   : {runs_processed}")
    print(f"Runs succeeded   : {runs_succeeded}")
    print(f"Runs failed      : {runs_failed}")

    if runs_failed > 0:
        print(
            f"\nWARNING: {runs_failed} run(s) failed AI enrichment. "
            f"Deterministic results are unaffected."
        )

    if runs_succeeded > 0:
        print(
            f"\nSUCCESS: {runs_succeeded} run(s) enriched with AI summaries."
        )

print(f"Model used       : {MODEL_ENDPOINT}")
print(f"Completed at     : {datetime.now()}")
print("=" * 60)